# Pipeline Lista Nominal de Hipertensão — passo a passo

Notebook exploratório que reproduz, célula a célula, o mesmo tratamento aplicado em `sql/01_bronze.sql` a `sql/04_metricas.sql`. A ideia é rodar camada por camada e ver o resultado de cada etapa (contagens, amostras, o funil de elegibilidade da lista) em vez de rodar tudo de uma vez.

**Como rodar:** a partir da pasta `notebooks/` (os caminhos dos CSVs abaixo são relativos a ela). Requer `duckdb` e `pandas` — e, para o gráfico opcional no final, `matplotlib`.

```
pip install duckdb pandas matplotlib jupyter
```

Este notebook não é um dos 5 entregáveis oficiais do case (esse é o SQL em `sql/`) — é material de apoio para explorar o mesmo pipeline interativamente.

In [ ]:
# Setup para Google Colab: instala o duckdb e clona o repositorio, se necessario.
# Em execucao local isso e pulado.

import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q duckdb
    import os
    if not os.path.isdir('case-impulsogov'):
        !git clone -q https://github.com/ShirleiAlexandrino/case-impulsogov.git
    %cd case-impulsogov/notebooks
    print('Rodando no Colab — repositorio clonado e duckdb instalado.')
else:
    print('Rodando localmente — nada a instalar.')

In [ ]:
# Importa as bibliotecas, confirma que os CSVs estao acessiveis e abre uma conexao
# DuckDB em memoria (tudo e recriado do zero a cada execucao, nada fica persistido).

from pathlib import Path

import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

assert Path('../dados/cidadao_pec.csv').exists(), (
    'CSVs não encontrados — confirme que está rodando a partir da pasta notebooks/ '
    'dentro do repositório do case.'
)

con = duckdb.connect(database=':memory:')
print('Conexão DuckDB em memória criada.')

## Camada Bronze — ingestão bruta

Espelho dos três CSVs, com tipagem explícita para não deixar o auto-detect do DuckDB promover códigos com zero à esquerda (`nu_ine`, `nu_cbo`, `nu_cns`, `nu_cpf`...) para tipo numérico e corrompê-los. Nenhuma regra de negócio é aplicada aqui ainda.

In [ ]:
# Le cidadao_pec.csv com tipagem explicita e cria bronze_cidadao_pec -- nenhuma
# limpeza ou regra de negocio ainda, e o espelho fiel do CSV de origem.

con.sql('''
CREATE OR REPLACE TABLE bronze_cidadao_pec AS
SELECT * FROM read_csv('../dados/cidadao_pec.csv',
    header = true,
    columns = {
        'co_fat_cidadao_pec':               'BIGINT',
        'no_cidadao':                       'VARCHAR',
        'dt_registro_nascimento':           'DATE',
        'ds_sexo':                          'VARCHAR',
        'nu_cns':                           'VARCHAR',
        'nu_cpf_cidadao':                   'VARCHAR',
        'nu_telefone_celular':              'VARCHAR',
        'st_faleceu':                       'SMALLINT',
        'dt_ultima_atualizacao_cidadao':    'TIMESTAMP',
        'st_diabetes_diagnosticada':        'SMALLINT',
        'st_hipertensao_diagnosticada':     'SMALLINT',
        'nu_ine':                           'VARCHAR',
        'no_equipe':                        'VARCHAR',
        'data_ultimo_atend_individual':     'DATE',
        'equipe_ine_atendimento':           'VARCHAR',
        'equipe_nome_atendimento':          'VARCHAR',
        'nu_atend_ubs_ultimos_12_meses':    'BIGINT',
        'data_transmissao':                 'DATE'
    }
);
''')

print('Linhas em bronze_cidadao_pec:', con.sql('SELECT count(*) FROM bronze_cidadao_pec').fetchone()[0])
con.sql('SELECT * FROM bronze_cidadao_pec LIMIT 5').df()

In [ ]:
# Mesma logica, agora para atendimento_individual.csv.

con.sql('''
CREATE OR REPLACE TABLE bronze_atendimento_individual AS
SELECT * FROM read_csv('../dados/atendimento_individual.csv',
    header = true,
    columns = {
        'co_seq_fat_atd_ind':      'VARCHAR',
        'co_fat_cidadao_pec':      'BIGINT',
        'atend_ind_classificacao': 'VARCHAR',
        'nu_ciap':                 'VARCHAR',
        'no_ciap':                 'VARCHAR',
        'nu_cid':                  'VARCHAR',
        'no_cid':                  'VARCHAR',
        'no_profissional':         'VARCHAR',
        'nu_cbo':                  'VARCHAR',
        'no_cbo':                  'VARCHAR',
        'ds_local_atendimento':    'VARCHAR',
        'nu_ine':                  'VARCHAR',
        'no_equipe':               'VARCHAR',
        'dt_registro':             'DATE',
        'data_transmissao':        'DATE',
        'propriedades':            'VARCHAR',
        'versao':                  'VARCHAR'
    }
);
''')

print('Linhas em bronze_atendimento_individual:', con.sql('SELECT count(*) FROM bronze_atendimento_individual').fetchone()[0])
con.sql('SELECT * FROM bronze_atendimento_individual LIMIT 5').df()

In [ ]:
# Mesma logica, agora para procedimentos.csv.

con.sql('''
CREATE OR REPLACE TABLE bronze_procedimentos AS
SELECT * FROM read_csv('../dados/procedimentos.csv',
    header = true,
    columns = {
        'tabela':                        'VARCHAR',
        'co_seq_fat_proced':              'VARCHAR',
        'co_fat_cidadao_pec':             'BIGINT',
        'co_proced':                      'VARCHAR',
        'ds_proced':                      'VARCHAR',
        'nu_ine':                         'VARCHAR',
        'no_equipe':                      'VARCHAR',
        'no_profissional':                'VARCHAR',
        'nu_cbo':                         'VARCHAR',
        'no_cbo':                         'VARCHAR',
        'dt_registro':                    'DATE',
        'qt_afericao_pressao_arterial':   'VARCHAR',
        'qt_glicemia':                    'VARCHAR',
        'data_transmissao':               'DATE',
        'propriedades':                   'VARCHAR',
        'versao':                         'VARCHAR'
    }
);
''')

print('Linhas em bronze_procedimentos:', con.sql('SELECT count(*) FROM bronze_procedimentos').fetchone()[0])
con.sql('SELECT * FROM bronze_procedimentos LIMIT 5').df()

## Camada Silver — deduplicação e limpeza

As cargas são mensais e incrementais: a mesma chave pode chegar em mais de uma transmissão (às vezes com pequenas divergências de texto, ex. `ESF 2` vs `Esf 2`). Aqui mantemos 1 linha por chave — a da transmissão mais recente — e extraímos os campos que vêm dentro do JSON `propriedades`.

In [ ]:
# Deduplica por co_fat_cidadao_pec mantendo a transmissao mais recente,
# e aplica trim() nos campos de texto (no_cidadao, no_equipe).

con.sql('''
CREATE OR REPLACE TABLE silver_cidadao AS
SELECT
    co_fat_cidadao_pec,
    trim(no_cidadao)          AS no_cidadao,
    dt_registro_nascimento,
    ds_sexo,
    nu_cns,
    nu_cpf_cidadao,
    nu_telefone_celular,
    st_faleceu = 1             AS faleceu,
    nu_ine,
    trim(no_equipe)            AS no_equipe,
    data_transmissao
FROM bronze_cidadao_pec
QUALIFY row_number() OVER (
    PARTITION BY co_fat_cidadao_pec
    ORDER BY data_transmissao DESC
) = 1;
''')

antes = con.sql('SELECT count(*) FROM bronze_cidadao_pec').fetchone()[0]
depois = con.sql('SELECT count(*) FROM silver_cidadao').fetchone()[0]
print(f'bronze: {antes} linhas -> silver: {depois} linhas (removidas {antes - depois} duplicatas de chave)')
con.sql('SELECT * FROM silver_cidadao LIMIT 5').df()

In [ ]:
# Deduplica por co_seq_fat_atd_ind e calcula cbo_familia (4 primeiros digitos
# de nu_cbo), usado depois para saber quem pode contar cada boa pratica.

con.sql('''
CREATE OR REPLACE TABLE silver_atendimento_individual AS
SELECT
    co_seq_fat_atd_ind,
    co_fat_cidadao_pec,
    atend_ind_classificacao,
    no_profissional,
    nu_cbo,
    left(nu_cbo, 4)            AS cbo_familia,
    dt_registro,
    data_transmissao
FROM bronze_atendimento_individual
QUALIFY row_number() OVER (
    PARTITION BY co_seq_fat_atd_ind
    ORDER BY data_transmissao DESC
) = 1;
''')

antes = con.sql('SELECT count(*) FROM bronze_atendimento_individual').fetchone()[0]
depois = con.sql('SELECT count(*) FROM silver_atendimento_individual').fetchone()[0]
print(f'bronze: {antes} linhas -> silver: {depois} linhas (removidas {antes - depois} duplicatas de chave)')
con.sql('SELECT * FROM silver_atendimento_individual LIMIT 5').df()

In [ ]:
# Deduplica por co_seq_fat_proced e extrai peso, altura e pressao arterial
# de dentro do JSON da coluna propriedades.

con.sql('''
CREATE OR REPLACE TABLE silver_procedimentos AS
SELECT
    co_seq_fat_proced,
    co_fat_cidadao_pec,
    ds_proced,
    no_profissional,
    nu_cbo,
    left(nu_cbo, 4)                                                  AS cbo_familia,
    dt_registro,
    try_cast(json_extract_string(propriedades, '$.peso')   AS DOUBLE) AS peso_kg,
    try_cast(json_extract_string(propriedades, '$.altura') AS DOUBLE) AS altura_cm,
    json_extract_string(propriedades, '$.pressao_arterial')           AS pressao_arterial,
    data_transmissao
FROM bronze_procedimentos
QUALIFY row_number() OVER (
    PARTITION BY co_seq_fat_proced
    ORDER BY data_transmissao DESC
) = 1;
''')

antes = con.sql('SELECT count(*) FROM bronze_procedimentos').fetchone()[0]
depois = con.sql('SELECT count(*) FROM silver_procedimentos').fetchone()[0]
print(f'bronze: {antes} linhas -> silver: {depois} linhas (removidas {antes - depois} duplicatas de chave)')
con.sql('SELECT * FROM silver_procedimentos WHERE peso_kg IS NOT NULL OR pressao_arterial IS NOT NULL LIMIT 5').df()

## Camada Gold — regra de negócio

Aqui é onde a regra de `contexto/02-regras-hipertensao.md` é aplicada: quem entra na lista, quem sai, e o status das três boas práticas. Construído em etapas — uma view por CTE do `sql/03_gold.sql` original — para dar pra ver o resultado de cada uma isoladamente.

In [ ]:
# Tres listas de referencia com os codigos CBO que contam para cada boa pratica
# (contexto/02-regras-hipertensao.md) -- usadas nos filtros das proximas celulas.

con.sql('''
CREATE OR REPLACE TABLE cbo_medico_enfermeiro AS
SELECT unnest(['2231','2251','2252','2253','2235']) AS cbo_familia;
''')
con.sql('''
CREATE OR REPLACE TABLE cbo_pressao AS
SELECT unnest(['2231','2251','2252','2253','2235','3222']) AS cbo_familia;
''')
con.sql('''
CREATE OR REPLACE TABLE cbo_peso_altura AS
SELECT unnest(['2231','2251','2252','2253','2235','3222','5151']) AS cbo_familia;
''')
print('Famílias de CBO habilitadas por boa prática, definidas em contexto/02-regras-hipertensao.md.')

In [ ]:
# Filtra os atendimentos classificados como hipertensao (ativa ou resolvida),
# feitos so por medico/enfermeiro -- e a materia-prima do criterio de entrada.

con.sql('''
CREATE OR REPLACE VIEW eventos_hipertensao AS
SELECT
    a.co_fat_cidadao_pec,
    a.dt_registro,
    a.atend_ind_classificacao
FROM silver_atendimento_individual a
WHERE a.atend_ind_classificacao IN ('HIPERTENSÃO', 'HIPERTENSÃO - CONDIÇÃO RESOLVIDA')
  AND a.cbo_familia IN (SELECT cbo_familia FROM cbo_medico_enfermeiro);
''')

print('Atendimentos de hipertensão por médico/enfermeiro:', con.sql('SELECT count(*) FROM eventos_hipertensao').fetchone()[0])
con.sql('SELECT * FROM eventos_hipertensao LIMIT 5').df()

In [ ]:
# Para cada pessoa, pega so o evento de hipertensao mais recente -- e ele que
# decide se a pessoa esta ativa ou com a condicao resolvida hoje.

con.sql('''
CREATE OR REPLACE VIEW status_mais_recente AS
SELECT
    co_fat_cidadao_pec,
    atend_ind_classificacao AS classificacao_mais_recente
FROM eventos_hipertensao
QUALIFY row_number() OVER (
    PARTITION BY co_fat_cidadao_pec
    ORDER BY dt_registro DESC
) = 1;
''')

print('Pessoas com pelo menos um evento qualificado:', con.sql('SELECT count(*) FROM status_mais_recente').fetchone()[0])
con.sql('SELECT classificacao_mais_recente, count(*) AS pessoas FROM status_mais_recente GROUP BY 1').df()

In [ ]:
# Aplica os dois criterios de saida: exclui quem faleceu e quem teve
# 'condicao resolvida' como evento mais recente. O que sobra entra na lista.

con.sql('''
CREATE OR REPLACE VIEW elegiveis AS
SELECT c.co_fat_cidadao_pec
FROM silver_cidadao c
JOIN status_mais_recente s ON s.co_fat_cidadao_pec = c.co_fat_cidadao_pec
WHERE NOT c.faleceu
  AND s.classificacao_mais_recente = 'HIPERTENSÃO';
''')

print('Pessoas elegíveis para a lista:', con.sql('SELECT count(*) FROM elegiveis').fetchone()[0])

### De onde vêm os números — o funil completo

Dos cidadãos do cadastro (`silver_cidadao`), quantos nunca tiveram um evento qualificado, quantos saíram por resolução, quantos saíram por óbito, e quantos sobraram elegíveis.

In [ ]:
funil = con.sql('''
SELECT
    (SELECT count(*) FROM silver_cidadao) AS total_cidadaos,
    (SELECT count(*) FROM status_mais_recente) AS com_evento_qualificado,
    (SELECT count(*) FROM silver_cidadao) - (SELECT count(*) FROM status_mais_recente) AS nunca_entrou,
    (SELECT count(*) FROM status_mais_recente WHERE classificacao_mais_recente = 'HIPERTENSÃO - CONDIÇÃO RESOLVIDA') AS saiu_por_resolucao,
    (
        SELECT count(*)
        FROM status_mais_recente s
        JOIN silver_cidadao c USING (co_fat_cidadao_pec)
        WHERE s.classificacao_mais_recente = 'HIPERTENSÃO' AND c.faleceu
    ) AS saiu_por_obito,
    (SELECT count(*) FROM elegiveis) AS elegiveis_finais
''').df()
funil

In [ ]:
# Data do atendimento mais recente (medico/enfermeiro, qualquer classificacao)
# -- usada para o status da boa pratica Consulta.

con.sql('''
CREATE OR REPLACE VIEW ultima_consulta AS
SELECT
    a.co_fat_cidadao_pec,
    max(a.dt_registro) AS dt_ultima_consulta
FROM silver_atendimento_individual a
WHERE a.cbo_familia IN (SELECT cbo_familia FROM cbo_medico_enfermeiro)
GROUP BY a.co_fat_cidadao_pec;
''')
con.sql('SELECT * FROM ultima_consulta LIMIT 5').df()

In [ ]:
# Leitura de pressao mais recente (medico, enfermeiro ou tecnico de enfermagem)
# -- traz tambem o valor, nao so a data, porque o prototipo exibe o valor.

con.sql('''
CREATE OR REPLACE VIEW ultima_afericao_pressao AS
SELECT
    p.co_fat_cidadao_pec,
    p.dt_registro       AS dt_ultima_afericao_pressao,
    p.pressao_arterial  AS valor_ultima_afericao_pressao
FROM silver_procedimentos p
WHERE p.ds_proced = 'AFERIÇÃO DE PRESSÃO ARTERIAL'
  AND p.cbo_familia IN (SELECT cbo_familia FROM cbo_pressao)
QUALIFY row_number() OVER (
    PARTITION BY p.co_fat_cidadao_pec
    ORDER BY p.dt_registro DESC
) = 1;
''')
con.sql('SELECT * FROM ultima_afericao_pressao LIMIT 5').df()

In [ ]:
# Data do registro de peso e altura mais recente, so quando as duas medidas
# vieram preenchidas no mesmo dia.

con.sql('''
CREATE OR REPLACE VIEW ultimo_peso_altura AS
SELECT
    p.co_fat_cidadao_pec,
    max(p.dt_registro) AS dt_ultimo_peso_altura
FROM silver_procedimentos p
WHERE p.ds_proced = 'MEDIÇÃO PESO E ALTURA'
  AND p.peso_kg IS NOT NULL
  AND p.altura_cm IS NOT NULL
  AND p.cbo_familia IN (SELECT cbo_familia FROM cbo_peso_altura)
GROUP BY p.co_fat_cidadao_pec;
''')
con.sql('SELECT * FROM ultimo_peso_altura LIMIT 5').df()

### Montagem final — `gold_lista_nominal_hipertensao`

Junta `elegiveis` + `silver_cidadao` + as três views de boa prática, calcula idade e classifica cada boa prática em EM DIA / ATRASADA / NUNCA REALIZADA. Data de referência do retrato: **2026-08-01** (fixa — não é `CURRENT_DATE`).

In [ ]:
con.sql('''
CREATE OR REPLACE TABLE gold_lista_nominal_hipertensao AS
SELECT
    c.co_fat_cidadao_pec,
    c.no_cidadao,
    c.nu_cns,
    c.nu_cpf_cidadao,
    c.dt_registro_nascimento,
    date_part('year', age(DATE '2026-08-01', c.dt_registro_nascimento)) AS idade,
    c.nu_telefone_celular,
    c.nu_ine,
    c.no_equipe,

    uc.dt_ultima_consulta,
    CASE
        WHEN uc.dt_ultima_consulta IS NULL THEN 'NUNCA REALIZADA'
        WHEN uc.dt_ultima_consulta >= DATE '2026-08-01' - INTERVAL 6 MONTH THEN 'EM DIA'
        ELSE 'ATRASADA'
    END AS status_consulta,

    up.dt_ultima_afericao_pressao,
    up.valor_ultima_afericao_pressao,
    CASE
        WHEN up.dt_ultima_afericao_pressao IS NULL THEN 'NUNCA REALIZADA'
        WHEN up.dt_ultima_afericao_pressao >= DATE '2026-08-01' - INTERVAL 6 MONTH THEN 'EM DIA'
        ELSE 'ATRASADA'
    END AS status_afericao_pressao,

    upa.dt_ultimo_peso_altura,
    CASE
        WHEN upa.dt_ultimo_peso_altura IS NULL THEN 'NUNCA REALIZADA'
        WHEN upa.dt_ultimo_peso_altura >= DATE '2026-08-01' - INTERVAL 12 MONTH THEN 'EM DIA'
        ELSE 'ATRASADA'
    END AS status_peso_altura

FROM elegiveis e
JOIN silver_cidadao c ON c.co_fat_cidadao_pec = e.co_fat_cidadao_pec
LEFT JOIN ultima_consulta uc          ON uc.co_fat_cidadao_pec  = c.co_fat_cidadao_pec
LEFT JOIN ultima_afericao_pressao up  ON up.co_fat_cidadao_pec  = c.co_fat_cidadao_pec
LEFT JOIN ultimo_peso_altura upa      ON upa.co_fat_cidadao_pec = c.co_fat_cidadao_pec;
''')

gold = con.sql('SELECT * FROM gold_lista_nominal_hipertensao').df()
print('Pessoas na lista final:', len(gold))
gold.head()

In [ ]:
# Confere a distribuicao de cada boa pratica direto no DataFrame pandas,
# sem precisar de outra query SQL.

for coluna in ['status_consulta', 'status_afericao_pressao', 'status_peso_altura']:
    print(f'--- {coluna} ---')
    print(gold[coluna].value_counts())
    print()

### Checagens rápidas de sanidade

Equivalentes simplificados aos guardrails de `sql/06_guardrails.sql`, só para confirmar visualmente que a tabela final está coerente.

In [ ]:
assert gold['co_fat_cidadao_pec'].is_unique, 'PK duplicada em gold — granularidade quebrada'
assert (gold['status_consulta'] != 'NUNCA REALIZADA').all(), 'violou a regra de entrada (consulta obrigatória)'

falecidos_na_lista = con.sql('''
    SELECT count(*) FROM gold_lista_nominal_hipertensao g
    JOIN silver_cidadao c USING (co_fat_cidadao_pec)
    WHERE c.faleceu
''').fetchone()[0]
assert falecidos_na_lista == 0, 'tem gente falecida na lista'

print('Todas as checagens passaram.')

## Métricas — números do entregável 2

Os mesmos números de `sql/04_metricas.sql`: total de pessoas na lista e a distribuição de status por boa prática.

In [ ]:
# Total de pessoas na lista final -- o primeiro numero pedido no entregavel 2.

total_pessoas = con.sql('SELECT count(*) AS total_pessoas_na_lista FROM gold_lista_nominal_hipertensao').df()
total_pessoas

In [ ]:
# Distribuicao de EM DIA / ATRASADA / NUNCA REALIZADA para cada boa pratica --
# o segundo numero pedido no entregavel 2.

distribuicao = con.sql('''
WITH distribuicao AS (
    SELECT 'Consulta' AS boa_pratica, status_consulta AS status, count(*) AS pessoas
    FROM gold_lista_nominal_hipertensao GROUP BY status_consulta
    UNION ALL
    SELECT 'Aferição de pressão', status_afericao_pressao, count(*)
    FROM gold_lista_nominal_hipertensao GROUP BY status_afericao_pressao
    UNION ALL
    SELECT 'Peso e altura', status_peso_altura, count(*)
    FROM gold_lista_nominal_hipertensao GROUP BY status_peso_altura
)
SELECT boa_pratica, status, pessoas
FROM distribuicao
ORDER BY 1,
    CASE status
        WHEN 'EM DIA' THEN 1
        WHEN 'ATRASADA' THEN 2
        WHEN 'NUNCA REALIZADA' THEN 3
    END;
''').df()
distribuicao

### Gráfico opcional

Requer `matplotlib` instalado — pule esta célula se não quiser essa dependência.

In [ ]:
import matplotlib.pyplot as plt

pivot = distribuicao.pivot(index='boa_pratica', columns='status', values='pessoas')
pivot = pivot[['EM DIA', 'ATRASADA', 'NUNCA REALIZADA']]
pivot.plot(kind='bar', stacked=True, figsize=(8, 5), color=['#1F8A70', '#EF8264', '#C13C2E'])
plt.title('Distribuição de status por boa prática')
plt.ylabel('Pessoas')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()